In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

np.random.seed(42)
base_temp = 9.5
amplitude = 14.0
city = "Київ"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp
        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"
        rows.append({
            "місто": city, "рік": year, "місяць": month,
            "температура": temp, "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows) # 48 рядків (4 роки x 12 місяців)
print(f"Перші 5 рядків датасету '{city}':")
print(climate.head())

Перші 5 рядків датасету 'Київ':
  місто   рік  місяць  температура  сезон відхилення_від_норми
0  Київ  2021       1         -4.0   зима            холодніше
1  Київ  2021       2         -2.8   зима            холодніше
2  Київ  2021       3          3.1  весна            холодніше
3  Київ  2021       4         11.0  весна             звичайно
4  Київ  2021       5         16.3  весна              тепліше


In [2]:
# 1. Абсолютні частоти (сирі кількості)
ct_abs = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
print("--- Таблиця спряженості (абсолютні частоти) ---")
print(ct_abs)

# 2. Нормована таблиця (відносні частоти по рядках)
ct_norm = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"], normalize="index")
print("\n--- Нормована таблиця спряженості (normalize='index') ---")
print(ct_norm.round(3))

--- Таблиця спряженості (абсолютні частоти) ---
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                        4        4          4
зима                         0        0         12
літо                         0       12          0
осінь                        4        4          4

--- Нормована таблиця спряженості (normalize='index') ---
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                    0.333    0.333      0.333
зима                     0.000    0.000      1.000
літо                     0.000    1.000      0.000
осінь                    0.333    0.333      0.333


In [3]:
chi2_stat, p_val, dof, expected = chi2_contingency(ct_abs)

df_expected = pd.DataFrame(expected, index=ct_abs.index, columns=ct_abs.columns)

print(f"Статистика Хі-квадрат (χ²): {chi2_stat:.4f}")
print(f"p-value: {p_val:.10e}")
print(f"Ступені свободи (dof): {dof}")
print("\nМатриця очікуваних частот (expected):")
print(df_expected.round(2))

Статистика Хі-квадрат (χ²): 38.4000
p-value: 9.3817041082e-07
Ступені свободи (dof): 6

Матриця очікуваних частот (expected):
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      5.0        5.0
зима                       2.0      5.0        5.0
літо                       2.0      5.0        5.0
осінь                      2.0      5.0        5.0


In [4]:
print("Чи всі очікувані частоти >= 5?")
print((df_expected >= 5).all().all())
print("\nОчікувані частоти клітинок:")
print(df_expected.round(2))

Чи всі очікувані частоти >= 5?
False

Очікувані частоти клітинок:
відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      5.0        5.0
зима                       2.0      5.0        5.0
літо                       2.0      5.0        5.0
осінь                      2.0      5.0        5.0


In [5]:
# Побудова агрегованого стовпця
climate["відхилення_агреговане"] = climate["відхилення_від_норми"].replace({
    "холодніше": "не_тепліше",
    "звичайно": "не_тепліше"
})

ct_merged = pd.crosstab(climate["сезон"], climate["відхилення_агреговане"])
chi2_m, p_val_m, dof_m, expected_m = chi2_contingency(ct_merged)
df_expected_m = pd.DataFrame(expected_m, index=ct_merged.index, columns=ct_merged.columns)

print("--- Агрегована таблиця спряженості ---")
print(ct_merged)
print("\nНові очікувані частоти (усі >= 5):")
print(df_expected_m)
print(f"\nНове p-value: {p_val_m:.10e}")

--- Агрегована таблиця спряженості ---
відхилення_агреговане  не_тепліше  тепліше
сезон                                     
весна                           8        4
зима                           12        0
літо                            0       12
осінь                           8        4

Нові очікувані частоти (усі >= 5):
відхилення_агреговане  не_тепліше  тепліше
сезон                                     
весна                         7.0      5.0
зима                          7.0      5.0
літо                          7.0      5.0
осінь                         7.0      5.0

Нове p-value: 9.2782407630e-06


In [6]:
observed_seasons = climate["сезон"].value_counts().reindex(["весна", "зима", "літо", "осінь"])
n_total = len(climate)

expected_seasons = [0.25 * n_total] * 4

res_seasons = chisquare(f_obs=observed_seasons, f_exp=expected_seasons)

print("Фактичні частоти сезонів:")
print(observed_seasons)
print(f"Статистика χ²: {res_seasons.statistic:.4f}")
print(f"p-value: {res_seasons.pvalue:.4f}")

Фактичні частоти сезонів:
сезон
весна    12
зима     12
літо     12
осінь    12
Name: count, dtype: int64
Статистика χ²: 0.0000
p-value: 1.0000


In [7]:
observed_norm = climate["відхилення_від_норми"].value_counts().reindex(["холодніше", "звичайно", "тепліше"])

expected_norm = [0.25 * n_total, 0.50 * n_total, 0.25 * n_total]  # [12, 24, 12]

res_norm = chisquare(f_obs=observed_norm, f_exp=expected_norm)

print("Спостережувані частоти категорій:")
print(observed_norm)
print(f"Очікувані частоти: {expected_norm}")
print(f"Статистика χ²: {res_norm.statistic:.4f}")
print(f"p-value: {res_norm.pvalue:.4f}")

Спостережувані частоти категорій:
відхилення_від_норми
холодніше    20
звичайно      8
тепліше      20
Name: count, dtype: int64
Очікувані частоти: [12.0, 24.0, 12.0]
Статистика χ²: 21.3333
p-value: 0.0000
